Bank Churn Prediction

## Problem Statement[¶](#Problem-Statement)

### Context[¶](#Context)

Businesses like banks which provide service have to worry about problem of 'Customer Churn' i.e. customers leaving and joining another service provider. It is important to understand which aspects of the service influence a customer's decision in this regard. Management can concentrate efforts on improvement of service, keeping in mind these priorities.

### Objective[¶](#Objective)

You as a Data scientist with the bank need to build a neural network based classifier that can determine whether a customer will leave the bank or not in the next 6 months.

### Data Dictionary[¶](#Data-Dictionary)

* CustomerId: Unique ID which is assigned to each customer
* Surname: Last name of the customer
* CreditScore: It defines the credit history of the customer.
* Geography: A customer’s location
* Gender: It defines the Gender of the customer
* Age: Age of the customer
* Tenure: Number of years for which the customer has been with the bank
* NumOfProducts: refers to the number of products that a customer has purchased through the bank.
* Balance: Account balance
* HasCrCard: It is a categorical variable which decides whether the customer has credit card or not.
* EstimatedSalary: Estimated salary
* isActiveMember: Is is a categorical variable which decides whether the customer is active member of the bank or not ( Active member in the sense, using bank products regularly, making transactions etc )
* Exited : whether or not the customer left the bank within six month. It can take two values
  \*\* 0=No ( Customer did not leave the bank )
  \*\* 1=Yes ( Customer left the bank )

## Importing necessary libraries[¶](#Importing-necessary-libraries)

In [ ]:
# Basic libraries
import numpy as np
import pandas as pd

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Data preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

# Handle class imbalance
from imblearn.over_sampling import SMOTE

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD, Adam

# Hyperparameter tuning
from sklearn.model_selection import GridSearchCV

## Loading the dataset[¶](#Loading-the-dataset)

In [ ]:
# Load the dataset
df = pd.read_csv("Churn.csv")

# Display first five rows
df.head()

## Data Overview[¶](#Data-Overview)

In [ ]:
# Shape of the dataset
print("Shape of the dataset:", df.shape)

# Information about the dataset
print("\nDataset Information:\n")
df.info()

# Statistical summary
print("\nStatistical Summary:\n")
display(df.describe(include="all"))

# Missing values
print("\nMissing Values:\n")
display(df.isnull().sum())

### Observations[¶](#Observations)

* The dataset contains **10,000 customer records** and **14 variables**, including the target variable **Exited**.
* The objective is to predict whether a customer will leave the bank within the next six months.
* The dataset contains a combination of numerical variables (e.g., CreditScore, Age, Balance, EstimatedSalary) and categorical variables (e.g., Geography, Gender).
* No missing values were found in any of the variables, so no imputation is required.
* Three variables (`RowNumber`, `CustomerId`, and `Surname`) appear to serve as identifiers rather than meaningful predictive features and will be evaluated before model build

In [ ]:
# Check the number of unique values in each column
df.nunique().sort_values(ascending=False)

### Identifier Removal[¶](#Identifier-Removal)

The variables **RowNumber**, **CustomerId**, and **Surname** were identified as identifier fields rather than meaningful customer attributes.

* **RowNumber** is simply the sequential index of each observation.
* **CustomerId** is a unique identifier assigned to each customer.
* **Surname** is a customer name with high cardinality and does not directly describe customer behavior.

Although **EstimatedSalary** has a very high number of unique values, it represents a meaningful financial characteristic of the customer and is therefore retained for analysis.

Based on these observations, **RowNumber**, **CustomerId**, and **Surname** are removed before model development.

In [ ]:
# Remove identifier columns
df = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

# Verify the remaining columns
df.head()

In [ ]:
df.columns

## Exploratory Data Analysis[¶](#Exploratory-Data-Analysis)

### Univariate Analysis[¶](#Univariate-Analysis)

In [ ]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=np.number).columns.drop("Exited")
cat_cols = df.select_dtypes(include="object").columns

# Numerical variables
for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

# Categorical variables
for col in cat_cols:
    plt.figure(figsize=(6,4))
    sns.countplot(data=df, x=col)
    plt.title(f"Count Plot of {col}")
    plt.xticks(rotation=45)
    plt.show()

# Target variable
plt.figure(figsize=(5,4))
sns.countplot(data=df, x="Exited")
plt.title("Distribution of Churn (Exited)")
plt.show()

### Univariate Analysis - Observations[¶](#Univariate-Analysis---Observations)

* Customer ages are concentrated around the middle-age range, with relatively few very young or very old customers.
* Credit scores appear approximately normally distributed.
* Customer balances vary considerably, with many customers having low or zero balances.
* Most customers own one or two banking products.
* France contains the largest share of customers, followed by Germany and Spain.
* Male and female customers are represented in similar proportions.
* The target variable is imbalanced, with substantially more customers staying with the bank than leaving.

### Bivariate Analysis[¶](#Bivariate-Analysis)

In [ ]:
# Relationship between numerical variables and churn
for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x="Exited", y=col, data=df)
    plt.title(f"{col} vs Exited")
    plt.show()

# Relationship between categorical variables and churn
for col in cat_cols:
    plt.figure(figsize=(6,4))
    sns.countplot(data=df, x=col, hue="Exited")
    plt.title(f"{col} vs Exited")
    plt.xticks(rotation=45)
    plt.show()

# Correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

### Bivariate Analysis - Observations[¶](#Bivariate-Analysis---Observations)

* Older customers generally exhibit a higher probability of churn than younger customers.
* Customers with higher account balances appear more likely to leave the bank.
* Active members tend to remain with the bank more frequently than inactive members.
* Churn rates differ across geographical regions, suggesting location influences customer retention.
* Customers owning multiple products generally show lower churn rates up to a certain point.
* The correlation matrix indicates that no pair of numerical variables exhibits severe multicollinearity, making them suitable for model development.

## Data Preprocessing[¶](#Data-Preprocessing)

### Dummy Variable Creation[¶](#Dummy-Variable-Creation)

In [ ]:
# Create dummy variables for categorical features
df = pd.get_dummies(df, drop_first=True)

# Display first five rows
df.head()

### Train-validation-test Split[¶](#Train-validation-test-Split)

In [ ]:
# Separate predictors and target
X = df.drop("Exited", axis=1)
y = df["Exited"]

# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)

### Data Normalization[¶](#Data-Normalization)

In [ ]:
# Standardize the predictor variables
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training data shape:", X_train.shape)
print("Testing data shape :", X_test.shape)

### Preprocessing Observations[¶](#Preprocessing-Observations)

* Categorical variables were converted into numerical dummy variables using one-hot encoding.
* The target variable (`Exited`) was separated from the predictor variables.
* The data was split into training and testing sets using stratified sampling to preserve the class distribution.
* Feature scaling was applied using StandardScaler since neural networks perform better when input features are on a similar scale.

## Model Building[¶](#Model-Building)

### Model Evaluation Criterion[¶](#Model-Evaluation-Criterion)

### Model Evaluation Criterion[¶](#Model-Evaluation-Criterion)

The dataset is imbalanced because significantly more customers remain with the bank than leave it. In this business problem, failing to identify a customer who is likely to churn (false negative) is generally more costly than incorrectly predicting that a customer will churn (false positive), since missed churners cannot be targeted with retention strategies.

Therefore, **Recall** is selected as the primary evaluation metric because it measures the model's ability to correctly identify customers who are likely to leave the bank. Accuracy, precision, F1-score, and ROC-AUC will also be reported to provide a comprehensive evaluation of model performance.

### Neural Network with SGD Optimizer[¶](#Neural-Network-with-SGD-Optimizer)

In [ ]:
# Build the neural network
model_sgd = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile the model
model_sgd.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the model
history_sgd = model_sgd.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Predict
y_pred_prob = model_sgd.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# Evaluation
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - SGD")
plt.show()

### Model Performance - SGD Optimizer[¶](#Model-Performance---SGD-Optimizer)

* The neural network trained successfully using the SGD optimizer.
* The model achieved high overall accuracy by correctly classifying the majority of non-churning customers.
* However, the recall for the churn class is relatively low, indicating that a considerable number of customers who actually left the bank were not identified.
* Since the business objective is to identify customers at risk of churn, improving recall is more important than maximizing overall accuracy.
* Therefore, additional models using the Adam optimizer, Dropout, hyperparameter tuning, and SMOTE will be evaluated to improve predictive performance.

## Model Performance Improvement[¶](#Model-Performance-Improvement)

### Neural Network with Adam Optimizer[¶](#Neural-Network-with-Adam-Optimizer)

In [ ]:
# Build the neural network
model_adam = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile the model using Adam optimizer
model_adam.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the model
history_adam = model_adam.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Predictions
y_pred_prob = model_adam.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# Performance metrics
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_pred_prob))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Adam Optimizer")
plt.show()

### Model Performance - Adam Optimizer[¶](#Model-Performance---Adam-Optimizer)

* Replacing SGD with the Adam optimizer resulted in faster convergence during training.
* The Adam optimizer generally provides better optimization by adapting the learning rate for each parameter.
* Model performance was evaluated using Accuracy, Precision, Recall, F1-score, and ROC-AUC.
* The results will be compared with the SGD model to determine whether Adam improves the identification of customers likely to churn.

### Neural Network with Adam Optimizer and Dropout[¶](#Neural-Network-with-Adam-Optimizer-and-Dropout)

In [ ]:
# Build the neural network with Dropout
model_dropout = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

# Compile
model_dropout.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train
history_dropout = model_dropout.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Predictions
y_pred_prob = model_dropout.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# Evaluation
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_pred_prob))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Adam + Dropout")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

### Model Performance - Adam with Dropout[¶](#Model-Performance---Adam-with-Dropout)

* Dropout layers were added to reduce overfitting by randomly deactivating neurons during training.
* The model was trained using the Adam optimizer and evaluated using Accuracy, Precision, Recall, F1-score, and ROC-AUC.
* The effect of dropout was assessed by comparing its performance with the standard Adam model.
* If validation performance improves while maintaining generalization, dropout can be considered beneficial for this classification problem.

### Neural Network with Balanced Data (by applying SMOTE) and SGD Optimizer[¶](#Neural-Network-with-Balanced-Data-(by-applying-SMOTE)-and-SGD-Optimizer)

In [ ]:
# Apply SMOTE to balance the training data
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Original training shape:", X_train.shape)
print("Balanced training shape:", X_train_smote.shape)

# Build the neural network
model_smote = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_smote.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile using SGD optimizer
model_smote.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the model
history_smote = model_smote.fit(
    X_train_smote,
    y_train_smote,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Predictions
y_pred_prob = model_smote.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# Evaluation
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_pred_prob))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - SMOTE + SGD")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

### Model Performance - Balanced Data using SMOTE and SGD[¶](#Model-Performance---Balanced-Data-using-SMOTE-and-SGD)

* SMOTE was applied to balance the minority class in the training data before model training.
* The neural network was trained using the SGD optimizer on the balanced dataset.
* Balancing the classes aims to improve the model's ability to identify customers who are likely to churn, particularly by increasing recall.
* The performance of this model will be compared with the previous models to select the best overall solution.

### Neural Network with Balanced Data (by applying SMOTE) and Adam Optimizer[¶](#Neural-Network-with-Balanced-Data-(by-applying-SMOTE)-and-Adam-Optimizer)

In [ ]:
# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Build model
model_smote_adam = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_smote.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile
model_smote_adam.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train
history_smote_adam = model_smote_adam.fit(
    X_train_smote,
    y_train_smote,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Predict
y_pred_prob = model_smote_adam.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_pred_prob))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("SMOTE + Adam")
plt.show()

### Observations[¶](#Observations)

* Applying SMOTE balanced the training data before model fitting.
* Adam optimizer was then used to train the neural network.
* The balanced dataset generally improves the model's ability to identify churning customers.
* This model will be compared with the previous models for final selection.

### Neural Network with Balanced Data (by applying SMOTE), Adam Optimizer, and Dropout[¶](#Neural-Network-with-Balanced-Data-(by-applying-SMOTE),-Adam-Optimizer,-and-Dropout)

In [ ]:
# Build model
model_smote_dropout = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_smote.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_smote_dropout.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_smote_dropout = model_smote_dropout.fit(
    X_train_smote,
    y_train_smote,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

y_pred_prob = model_smote_dropout.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_pred_prob))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("SMOTE + Adam + Dropout")
plt.show()

### Observations[¶](#Observations)

* SMOTE, Adam optimizer, and Dropout were combined to improve model generalization.
* Dropout helps reduce overfitting while SMOTE addresses class imbalance.
* This model represents one of the most comprehensive approaches evaluated in this project.

## Model Performance Comparison and Final Model Selection[¶](#Model-Performance-Comparison-and-Final-Model-Selection)

## Model Performance Comparison and Final Model Selection[¶](#Model-Performance-Comparison-and-Final-Model-Selection)

Several neural network models were evaluated, including SGD, Adam, Adam with Dropout, SMOTE with SGD, SMOTE with Adam, and SMOTE with Adam and Dropout.

Among these models, the neural network trained using **SMOTE and the Adam optimizer** was selected as the final model.

Although the SMOTE + Adam + Dropout model achieved a slightly higher ROC-AUC score, the SMOTE + Adam model correctly identified more customers who were likely to churn while producing fewer false negatives. Since the primary business objective is to identify customers at risk of leaving the bank, recall was considered the most important evaluation metric.

Therefore, the SMOTE + Adam model was selected because it provides the best balance between recall, precision, F1-score, and overall predictive performance, making it the most appropriate model for customer retention strategies.

## Actionable Insights and Business Recommendations[¶](#Actionable-Insights-and-Business-Recommendations)

# Actionable Insights and Recommendations[¶](#Actionable-Insights-and-Recommendations)

* Older customers and inactive members appear more likely to leave the bank, suggesting they should be prioritized for retention campaigns.
* Customers with higher account balances represent valuable clients and should receive personalized offers to reduce churn risk.
* The trained neural network can be integrated into the bank's customer relationship management system to identify customers who are at high risk of leaving.
* Customers predicted to churn can be targeted with loyalty programs, personalized financial products, or proactive customer support.
* The model should be retrained periodically using updated customer data to maintain prediction accuracy as customer behavior changes over time.

Power Ahead

---